# Figure 3: Biological hierarchy

**Paper:** BATTLE-AMP

**Per-panel stories:**
- **(a)** Dataset size shrinks and class imbalance shifts as the biological target narrows from broad activity down to individual strains.
- **(b)** No single model dominates across all biological targets; performance is strongly target-dependent, with species-specific models winning only on their matched target.
- **(c)** Regression models show moderate rank correlations but poor calibration, and MBC-Attention is the only model with positive R-squared on E. coli.

**Layout:** 3 stacked rows. Panel a (stacked bars). Panel b (MCC heatmap 13x15). Panel c (regression heatmap 7x6).


In [4]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Paths
# ============================================================
SUMMARY_FILE = Path("../results/aggregated/summary.tsv")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(exist_ok=True)
# ============================================================
# Figure constants (Briefings in Bioinformatics)
# ============================================================
MAX_WIDTH = 6.5
MAX_HEIGHT = 8.0
TARGET_DPI = 600

CLF_COLOR = "#d95f6e"   # muted rose      -- classifiers
ACT_COLOR = "#9467bd"   # muted purple    -- HydrAMP-MIC
REG_COLOR = "#8c8c8c"   # medium grey     -- regressors

GRID_COLOR = "#dddddd"
FONTSIZE_TICK = 6
FONTSIZE_LABEL = 7
FONTSIZE_PANEL = 10
FONTSIZE_ANNOT = 5

# ============================================================
# Biological hierarchy: tasks
# ============================================================
CLF_TASKS = [
    "broad_activity",
    "gram_minus", "gram_plus",
    "species_ecoli", "species_saureus", "species_paeruginosa",
    "species_kpneumoniae", "species_abaumannii",
    "strain_ecoli25922",
    "strain_saureus25923", "strain_saureus33591", "strain_saureus43300",
    "strain_paeruginosa27853",
    "strain_kpneumoniae700603",
    "strain_abaumannii19606",
]
CLF_TASK_SHORT = [
    "General\nActivity",
    "Gram$-$", "Gram$+$",
    "EC", "SA", "PA", "KP", "AB",
    "EC\n25922", "SA\n25923", "SA\n33591", "SA\n43300",
    "PA\n27853", "KP\n700603", "AB\n19606",
]

TASK_GROUPS = [
    ("General", 0, 1),
    ("Gram", 1, 3),
    ("Species", 3, 8),
    ("Strain", 8, 15),
]

# ============================================================
# Models (matching Figure 2 conventions)
# ============================================================
CLASSIFIERS = [
    "hydramp-amp-classifier", "ampscanner", "amplify",
    "ampeppy", "ampredmfa",
]
ACTIVITY_AWARE = ["hydramp-mic-classifier"]
REGRESSORS = [
    "mbc-attention",
    "apex-ecoli", "apex-saureus", "apex-min",
    "apex-abaumannii", "apex-paeruginosa", "apex-kpneumoniae",
]
ALL_CLF_MODELS = CLASSIFIERS + ACTIVITY_AWARE + REGRESSORS

MODEL_DISPLAY = {
    "hydramp-amp-classifier": "HydrAMP$_{AMP}$",
    "ampscanner": "AMP Scanner$_2$",
    "amplify": "AMPlify",
    "ampeppy": "amPEPpy",
    "ampredmfa": "AMPpred-MFA",
    "hydramp-mic-classifier": "HydrAMP$_{MIC}$",
    "mbc-attention": "MBC-Attention",
    "apex-ecoli": "APEX$_{EC}$",
    "apex-saureus": "APEX$_{SA}$",
    "apex-min": "APEX$_{min}$",
    "apex-abaumannii": "APEX$_{AB}$",
    "apex-paeruginosa": "APEX$_{PA}$",
    "apex-kpneumoniae": "APEX$_{KP}$",
}

MODEL_GROUPS = [
    ("Classifiers", 0, len(CLASSIFIERS) + len(ACTIVITY_AWARE)),
    ("Regressors\n(binarized)", len(CLASSIFIERS) + len(ACTIVITY_AWARE),
     len(ALL_CLF_MODELS)),
]


def model_color(m):
    if m in CLASSIFIERS:
        return CLF_COLOR
    elif m in ACTIVITY_AWARE:
        return ACT_COLOR
    return REG_COLOR


def short_name(m):
    return MODEL_DISPLAY.get(m, m)


# ============================================================
# Regression panel config
# ============================================================
REG_MODELS = [
    "mbc-attention",
    "apex-ecoli", "apex-saureus", "apex-min",
    "apex-abaumannii", "apex-paeruginosa", "apex-kpneumoniae",
]
REG_TASKS = ["regression_ecoli25922", "regression_saureus25923"]
REG_TASK_LABELS = ["EC ATCC 25922", "SA ATCC 25923"]
REG_METRICS = ["r2_log2", "spearman", "msl2e"]
REG_METRIC_LABELS = ["$R^2_{\\log_2}$", "$\\rho$", "MSL2E"]
HIGHER_IS_BETTER = [True, True, False]

# ============================================================
# Load data
# ============================================================
df = pd.read_csv(SUMMARY_FILE, sep="\t")
df = df[df["variant"] != "example-model"].copy()


def build_matrix(models, tasks, metric):
    mat = np.full((len(models), len(tasks)), np.nan)
    for i, m in enumerate(models):
        for j, t in enumerate(tasks):
            row = df[(df["variant"] == m) & (df["task"] == t)]
            if not row.empty:
                val = row.iloc[0][metric]
                if pd.notna(val) and val != "":
                    try:
                        mat[i, j] = float(val)
                    except (ValueError, TypeError):
                        pass
    return mat


clf_mcc = build_matrix(ALL_CLF_MODELS, CLF_TASKS, "mcc")
clf_cov = build_matrix(ALL_CLF_MODELS, CLF_TASKS, "coverage")

# Dataset stats per task
task_stats = {}
for task in CLF_TASKS:
    rows = df[(df["task"] == task) & (df["coverage"] >= 0.99)]
    if rows.empty:
        rows = df[df["task"] == task].sort_values("coverage", ascending=False)
    if not rows.empty:
        r = rows.iloc[0]
        n_pos = int(float(r["n_positive"]))
        n_neg = int(float(r["n_negative"]))
        task_stats[task] = {"n": n_pos + n_neg, "n_pos": n_pos, "n_neg": n_neg,
                            "pos_frac": n_pos / (n_pos + n_neg)}

print(f"Classification models: {len(ALL_CLF_MODELS)}")
print(f"Classification tasks: {len(CLF_TASKS)}")
print(f"Regression models: {len(REG_MODELS)}")

# ============================================================
# Global style
# ============================================================
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [
    "Helvetica", "Arial", "Liberation Sans", "DejaVu Sans",
]
matplotlib.rcParams["mathtext.default"] = "regular"
matplotlib.rcParams["axes.linewidth"] = 0.5
matplotlib.rcParams["xtick.major.width"] = 0.4
matplotlib.rcParams["ytick.major.width"] = 0.4



Classification models: 13
Classification tasks: 15
Regression models: 7


In [5]:
# ============================================================
# Build figure
# ============================================================
fig = plt.figure(figsize=(MAX_WIDTH, MAX_HEIGHT), dpi=TARGET_DPI,
                 facecolor="white")

# Three separate GridSpecs for precise control
gs_a = gridspec.GridSpec(1, 1, left=0.20, right=0.88,
                         top=0.97, bottom=0.855)
gs_b = gridspec.GridSpec(1, 1, left=0.20, right=0.88,
                         top=0.82, bottom=0.36)
gs_c = gridspec.GridSpec(1, 1, left=0.20, right=0.88,
                         top=0.24, bottom=0.04)

# ================================================================
# Panel A: Dataset sizes (stacked bar)
# ================================================================
ax_a = fig.add_subplot(gs_a[0, 0])
ax_a.set_facecolor("white")

n_tasks = len(CLF_TASKS)
x_a = np.arange(n_tasks)
bar_w = 0.7

n_pos_vals = [task_stats[t]["n_pos"] for t in CLF_TASKS]
n_neg_vals = [task_stats[t]["n_neg"] for t in CLF_TASKS]
n_totals = [task_stats[t]["n"] for t in CLF_TASKS]

ax_a.bar(x_a, n_pos_vals, bar_w, label="Active", color="#4daf4a",
         edgecolor="white", linewidth=0.3)
ax_a.bar(x_a, n_neg_vals, bar_w, bottom=n_pos_vals, label="Inactive",
         color="#e41a1c", edgecolor="white", linewidth=0.3)

for i, n in enumerate(n_totals):
    ax_a.text(i, n + 20, str(n), ha="center", va="bottom", fontsize=FONTSIZE_TICK)

ax_a.set_ylabel("Peptides", fontsize=FONTSIZE_LABEL)
ax_a.set_xticks(x_a)
ax_a.set_xticklabels([])  # shared axis with panel b below
ax_a.tick_params(axis="x", length=0)
ax_a.tick_params(axis="y", labelsize=FONTSIZE_TICK)
ax_a.set_xlim(-0.5, n_tasks - 0.5)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

ax_a.yaxis.set_major_locator(plt.MultipleLocator(1000))
ax_a.yaxis.grid(True, color=GRID_COLOR, linewidth=0.5)
ax_a.set_axisbelow(True)

ax_a.legend(fontsize=FONTSIZE_TICK, loc="upper right", frameon=True,
            facecolor="white", edgecolor="#cccccc", ncol=2)

# x-axis labels carried by panel b's heatmap below


# ================================================================
# Panel B: MCC heatmap
# ================================================================
ax_b = fig.add_subplot(gs_b[0, 0])

# Mask: NaN or coverage < 0.5
mask_b = np.isnan(clf_mcc) | ((~np.isnan(clf_cov)) & (clf_cov < 0.5))
norm_b = TwoSlopeNorm(vmin=-0.2, vcenter=0, vmax=0.75)

sns.heatmap(
    clf_mcc,
    mask=mask_b,
    ax=ax_b,
    cmap="RdYlGn",
    norm=norm_b,
    linewidths=0.2,
    linecolor="white",
    annot=True,
    fmt=".2f",
    annot_kws={"size": FONTSIZE_ANNOT},
    cbar_kws={"shrink": 0.5, "aspect": 20, "pad": 0.02},
    xticklabels=CLF_TASK_SHORT,
    yticklabels=[short_name(m) for m in ALL_CLF_MODELS],
)

# Grey fill for masked cells
n_models = len(ALL_CLF_MODELS)
for i in range(n_models):
    for j in range(n_tasks):
        if mask_b[i, j]:
            ax_b.add_patch(mpatches.Rectangle(
                (j, i), 1, 1, facecolor="#f0f0f0", edgecolor="white",
                linewidth=0.2))

ax_b.set_xticklabels(ax_b.get_xticklabels(), rotation=45, ha="right",
                     fontsize=FONTSIZE_TICK)
ax_b.xaxis.set_ticks_position("bottom")

# Color y-tick labels by model type
ytick_labels = ax_b.get_yticklabels()
for lbl in ytick_labels:
    lbl.set_fontsize(FONTSIZE_TICK)
    text = lbl.get_text()
    for m in ALL_CLF_MODELS:
        if short_name(m) == text:
            lbl.set_color(model_color(m))
            break
ax_b.set_yticklabels(ytick_labels)
ax_b.tick_params(axis="both", length=0)

# Colorbar
cbar_b = ax_b.collections[0].colorbar
cbar_b.ax.tick_params(labelsize=FONTSIZE_TICK)
cbar_b.set_label("MCC", fontsize=FONTSIZE_LABEL)

# Model group separator
for label, start, end in MODEL_GROUPS:
    if start > 0:
        ax_b.axhline(y=start, color="black", linewidth=1.0)

# Task group separators (vertical)
for _, _, end in TASK_GROUPS[:-1]:
    ax_b.axvline(x=end, color="black", linewidth=0.5, alpha=0.6)

# Task group labels above heatmap
for label, start, end in TASK_GROUPS:
    mid = (start + end) / 2.0 / n_tasks
    ax_b.text(mid, 1.01, label, ha="center", va="bottom",
              fontsize=FONTSIZE_TICK, transform=ax_b.transAxes)

# Model group labels in left margin
for label, start, end in MODEL_GROUPS:
    mid_frac = 1.0 - (start + end) / 2.0 / n_models
    ax_b.text(
        -0.28, mid_frac, label.replace("\n", " "),
        ha="center", va="center", fontsize=FONTSIZE_TICK, rotation=90,
        transform=ax_b.transAxes, clip_on=False,
    )


# ================================================================
# Panel C: Regression heatmap
# ================================================================
ax_c = fig.add_subplot(gs_c[0, 0])

n_orgs = len(REG_TASKS)
n_mets = len(REG_METRICS)
reg_raw = np.full((len(REG_MODELS), n_orgs * n_mets), np.nan)

for i, m in enumerate(REG_MODELS):
    for j, t in enumerate(REG_TASKS):
        for k, metric in enumerate(REG_METRICS):
            row = df[(df["variant"] == m) & (df["task"] == t)]
            if not row.empty:
                val = row.iloc[0][metric]
                if pd.notna(val) and val != "":
                    try:
                        reg_raw[i, j * n_mets + k] = float(val)
                    except (ValueError, TypeError):
                        pass

# Normalize per column for coloring
reg_norm = reg_raw.copy()
for col_idx in range(reg_raw.shape[1]):
    metric_idx = col_idx % n_mets
    col = reg_raw[:, col_idx]
    valid = col[~np.isnan(col)]
    if len(valid) > 0:
        vmin, vmax = valid.min(), valid.max()
        if vmax > vmin:
            reg_norm[:, col_idx] = (col - vmin) / (vmax - vmin)
            if not HIGHER_IS_BETTER[metric_idx]:
                reg_norm[:, col_idx] = 1.0 - reg_norm[:, col_idx]
        else:
            reg_norm[:, col_idx] = 0.5

# Annotation text
annot_text = np.full(reg_raw.shape, "", dtype=object)
for i in range(reg_raw.shape[0]):
    for j in range(reg_raw.shape[1]):
        if not np.isnan(reg_raw[i, j]):
            metric_idx = j % n_mets
            if metric_idx == 2:  # MSL2E
                annot_text[i, j] = f"{reg_raw[i, j]:.1f}"
            else:
                annot_text[i, j] = f"{reg_raw[i, j]:.2f}"

mask_c = np.isnan(reg_raw)

col_labels = []
for _ in REG_TASK_LABELS:
    for m_label in REG_METRIC_LABELS:
        col_labels.append(m_label)

sns.heatmap(
    reg_norm,
    mask=mask_c,
    ax=ax_c,
    cmap="RdYlGn",
    vmin=0, vmax=1,
    linewidths=0.2,
    linecolor="white",
    annot=annot_text,
    fmt="",
    annot_kws={"size": FONTSIZE_ANNOT},
    cbar=False,
    xticklabels=col_labels,
    yticklabels=[short_name(m) for m in REG_MODELS],
)

ax_c.set_xticklabels(ax_c.get_xticklabels(), rotation=0, ha="center",
                     fontsize=FONTSIZE_TICK)

# Color y-tick labels by model type
ytick_labels_c = ax_c.get_yticklabels()
for lbl in ytick_labels_c:
    lbl.set_fontsize(FONTSIZE_TICK)
    text = lbl.get_text()
    for m in REG_MODELS:
        if short_name(m) == text:
            lbl.set_color(model_color(m))
            break
ax_c.set_yticklabels(ytick_labels_c)
ax_c.tick_params(axis="both", length=0)

# Separator between organisms
ax_c.axvline(x=n_mets, color="black", linewidth=1.0)

# Organism labels above
for idx, org_label in enumerate(REG_TASK_LABELS):
    mid_x = (idx * n_mets + n_mets / 2.0) / (n_mets * n_orgs)
    ax_c.text(mid_x, 1.08, org_label, ha="center", va="bottom",
              fontsize=FONTSIZE_LABEL, transform=ax_c.transAxes)

# Colorbar for panel C
ax_c_pos = ax_c.get_position()
cax = fig.add_axes([
    ax_c_pos.x1 + 0.02,
    ax_c_pos.y0,
    0.012,
    ax_c_pos.height,
])
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(0, 1))
sm.set_array([])
cb = fig.colorbar(sm, cax=cax)
cb.set_ticks([0, 0.5, 1])
cax.set_yticklabels(["Worst", "", "Best"])
cax.tick_params(labelsize=FONTSIZE_TICK, size=2)
cb.set_label("Per-column rank", fontsize=FONTSIZE_TICK, labelpad=2)


# ================================================================
# Panel labels
# ================================================================
fig.text(0.07, 0.97, "a", fontsize=FONTSIZE_PANEL, fontweight="bold",
         va="top")
fig.text(0.07, 0.82, "b", fontsize=FONTSIZE_PANEL, fontweight="bold",
         va="top")
fig.text(0.07, 0.24, "c", fontsize=FONTSIZE_PANEL, fontweight="bold",
         va="top")


# ================================================================
# Save
# ================================================================
out_pdf = FIGURE_DIR / "figure3_hierarchy.pdf"
out_png = FIGURE_DIR / "figure3_hierarchy.png"
fig.savefig(str(out_pdf), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
fig.savefig(str(out_png), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
print(f"\nSaved {out_pdf} and {out_png}")
plt.close(fig)



Saved ../figures/figure3_hierarchy.pdf and ../figures/figure3_hierarchy.png


## Alt text

**Figure 3.** Biological hierarchy evaluation of AMP prediction models.
**(a)** Stacked bar chart showing dataset sizes (active in green, inactive in red)
for 15 classification tasks spanning broad activity, gram type, species, and strain
levels. Datasets shrink from 4,355 (GeneralActivity) to 98 (SA 43300).
**(b)** MCC heatmap for 13 models (6 classifiers + HydrAMP-MIC + 7 regressors,
binarized) across all 15 tasks. Color scale from red (negative MCC) through yellow
(zero) to green (MCC 0.7). Grey cells indicate models with <50% coverage or missing
predictions. MBC-Attention leads on broad and gram-level tasks. APEX-AB achieves the
highest single-cell MCC (0.55) on SA strain 25923. Classifiers show uniformly low MCC
across all tasks. Horizontal line separates classifiers from regressors; vertical lines
separate task groups.
**(c)** Regression heatmap for 7 regressors on E. coli ATCC 25922 and S. aureus ATCC
25923, showing R-squared (log2), Spearman correlation, and MSL2E. Per-column rank
normalization (green = best, red = worst). MBC-Attention is the only model with
positive R-squared (0.23 on E. coli). All APEX variants show negative R-squared,
indicating predictions worse than a constant mean.
